# 03. Where Are People? Spatial Hotspots at 500 m

**Data source:** `presence_cell500m_daily.csv` (5,715 grid cells x 151 days)

**Questions**
1. Where is device presence concentrated across DIY?
2. Are there suspicious "super-cells" that look like data artefacts rather than real crowds?
3. Which places become relatively busier during the Christmas and New Year holidays (tourism)?

In [ ]:
# @title 0. Setup (jalankan sekali, otomatis di Google Colab)
# Di Colab: membuat folder kerja, menulis modul helper, dan mengunduh data dari Zenodo.
# Di laptop (repo GitHub): sel ini tidak melakukan apa-apa.
import os, sys, hashlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE = "/content/jogjapulse"
    for d in ["src", "data", "figures", "notebooks"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    open(f"{BASE}/src/__init__.py", "w").close()
    with open(f"{BASE}/src/jogjapulse.py", "w") as f:
        f.write('"""\nShared helpers for the JogjaPulse human mobility project.\n\nAll notebooks import from here so that loading, cleaning and normalisation\nare done the same way everywhere.\n\nDataset: Widyawan et al. (2026). JogjaPulse: An Aggregated Spatio-Temporal\nMobility Dataset for Yogyakarta, Indonesia. Zenodo.\nhttps://doi.org/10.5281/zenodo.23157460  (CC BY 4.0)\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------\n# Paths\n# --------------------------------------------------------------------------\nROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = ROOT / "data"\nFIG_DIR = ROOT / "figures"\nFIG_DIR.mkdir(exist_ok=True)\n\n# --------------------------------------------------------------------------\n# Constants from the dataset README\n# --------------------------------------------------------------------------\nGOOD_PERIOD_END = "2022-02-28"      # OD flows after this are reduced quality\nSAMPLE_BREAKS = ["2022-01-01", "2022-02-16"]  # abrupt drops in device sample\nANOMALY_DAYS = ["2022-03-08"]       # one-day delivery issue\nNIGHT_HOURS = list(range(0, 6))     # 00:00-05:59 WIB\nDAY_HOURS = list(range(10, 17))     # 10:00-16:59 WIB\nHOLIDAY_WINDOW = ("2021-12-24", "2022-01-02")  # Christmas and New Year\nKAB_ORDER = ["YOGYAKARTA", "SLEMAN", "BANTUL", "KULON PROGO", "GUNUNGKIDUL"]\nDOW_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]\n\n\ndef _path(name: str) -> Path:\n    p = DATA_DIR / name\n    if not p.exists():\n        raise FileNotFoundError(\n            f"{p} not found. Run `python scripts/download_data.py` first "\n            "or download the CSV files from Zenodo into the data/ folder."\n        )\n    return p\n\n\n# --------------------------------------------------------------------------\n# Loaders\n# --------------------------------------------------------------------------\ndef load_hourly(drop_anomalies: bool = True) -> pd.DataFrame:\n    """Kapanewon x hour presence (n_maids, n_pings)."""\n    df = pd.read_csv(_path("presence_kapanewon_hourly.csv"),\n                     dtype={"kab_code": str, "kapanewon_code": str})\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    df["dow"] = df["date"].dt.dayofweek          # 0 = Monday\n    df["is_weekend"] = df["dow"] >= 5\n    df["month"] = df["date"].dt.to_period("M").astype(str)\n    return df\n\n\ndef load_cells(drop_anomalies: bool = True) -> pd.DataFrame:\n    """500 m grid cell x day presence (n_maids)."""\n    df = pd.read_csv(_path("presence_cell500m_daily.csv"))\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    return df\n\n\ndef load_od() -> pd.DataFrame:\n    """Monthly home-based OD visitor counts, joined with origin denominators."""\n    od = pd.read_csv(_path("od_home_visits_kapanewon_monthly.csv"),\n                     dtype={"origin_code": str, "dest_code": str,\n                            "origin_kab_code": str, "dest_kab_code": str})\n    org = pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                      dtype={"origin_code": str, "origin_kab_code": str})\n    od = od.merge(org[["month", "origin_code", "n_homed_maids",\n                       "n_homed_maids_daytime"]],\n                  on=["month", "origin_code"], how="left")\n    od["share"] = od["n_maids"] / od["n_homed_maids_daytime"]\n    od["is_internal"] = od["origin_code"] == od["dest_code"]\n    od["reduced_quality"] = od["reduced_quality"].astype(str).str.lower() == "true"\n    return od\n\n\ndef load_origins() -> pd.DataFrame:\n    return pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                       dtype={"origin_code": str, "origin_kab_code": str})\n\n\ndef load_home_vs_population() -> pd.DataFrame:\n    return pd.read_csv(_path("home_maids_vs_population_kapanewon.csv"))\n\n\ndef load_night_validation() -> pd.DataFrame:\n    return pd.read_csv(_path("validation_night_presence_kapanewon.csv"))\n\n\n# --------------------------------------------------------------------------\n# Normalisation (README section 6: the device sample shrinks over time,\n# so raw counts must never be compared across dates)\n# --------------------------------------------------------------------------\ndef add_hourly_share(df: pd.DataFrame) -> pd.DataFrame:\n    """Share of each kapanewon in the regional total of the same hour.\n\n    Note: n_maids is a distinct count, so the regional sum is not the exact\n    number of distinct devices. It is used here only as a common denominator\n    that removes the effect of the shrinking sample.\n    """\n    out = df.copy()\n    tot = out.groupby(["date", "hour"])["n_maids"].transform("sum")\n    out["share_hour"] = out["n_maids"] / tot\n    return out\n\n\ndef add_daily_share(cells: pd.DataFrame) -> pd.DataFrame:\n    """Share of each grid cell in the total presence of the same day."""\n    out = cells.copy()\n    out["share_day"] = out["n_maids"] / out.groupby("date")["n_maids"].transform("sum")\n    return out\n\n\ndef hourly_profile(df: pd.DataFrame, value: str = "n_pings") -> pd.DataFrame:\n    """Percentage of each day\'s activity that falls in each hour.\n\n    n_pings is additive, so it is the correct variable for regional totals.\n    Returns one row per date with 24 hour columns.\n    """\n    daily = df.groupby(["date", "hour"])[value].sum().unstack("hour").fillna(0)\n    return daily.div(daily.sum(axis=1), axis=0) * 100\n\n\ndef kapanewon_profiles(df: pd.DataFrame) -> pd.DataFrame:\n    """24-hour shape per kapanewon (each row sums to 1), based on the mean\n    hourly share. Used for functional-zone clustering."""\n    s = add_hourly_share(df)\n    prof = s.groupby(["kapanewon", "hour"])["share_hour"].mean().unstack("hour")\n    prof = prof.reindex(columns=range(24))\n    prof = prof.interpolate(axis=1, limit_direction="both")\n    return prof.div(prof.sum(axis=1), axis=0)\n\n\ndef savefig(fig, name: str, dpi: int = 200) -> Path:\n    path = FIG_DIR / name\n    fig.savefig(path, dpi=dpi, bbox_inches="tight")\n    return path\n')
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "plotly>=5.24"], check=False)
    import requests
    FILES = {'presence_cell500m_daily.csv': '0b4d544e8c9ab26d69360821e9ebff27', 'presence_kapanewon_hourly.csv': '90a4a8ae6eb68abc68f7a1021a51cce8', 'od_home_visits_kapanewon_monthly.csv': '5857291c608dd5335f6d8833c1ec7b7a', 'od_origin_homes_monthly.csv': 'd14368c48d3937c9146ed9054aee3c79', 'od_summary.csv': '139962d1c8101eb6873be6f902320efb', 'home_maids_vs_population_kapanewon.csv': '0f01f7a3bb7027174cc1b5e1ad0e8224', 'validation_night_presence_kapanewon.csv': '4a8d0f393fd6d4fa17fb0e60c6f7408e', 'tier1_summary.csv': 'd391339521ce3714f1616c430512b0e0'}
    for name, md5 in FILES.items():
        target = f"{BASE}/data/{name}"
        if os.path.exists(target):
            continue
        print("Downloading", name, "...", end=" ", flush=True)
        r = requests.get("https://zenodo.org/records/23157460/files/" + name, params={"download": 1}, timeout=300)
        r.raise_for_status()
        open(target, "wb").write(r.content)
        print("OK" if hashlib.md5(r.content).hexdigest() == md5 else "checksum mismatch")
    os.chdir(f"{BASE}/notebooks")
    print("Setup selesai. Folder kerja:", BASE)


In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

from src import jogjapulse as jp

sns.set_theme(style="white", context="talk", font_scale=0.8)
cells = jp.add_daily_share(jp.load_cells())
n_days = cells["date"].nunique()
print(cells.shape, "| cells:", cells["cell_id"].nunique(), "| days:", n_days)

## 3.1 Average share of daily presence per cell
Missing cell-days (suppressed, fewer than 5 devices) are treated as share 0 when averaging,
which is a conservative lower bound.

In [ ]:
agg = (cells.groupby(["cell_id", "centroid_lon", "centroid_lat"])
       .agg(share_sum=("share_day", "sum"), days_published=("date", "nunique"))
       .reset_index())
agg["mean_share_bp"] = agg["share_sum"] / n_days * 1e4     # basis points (1 bp = 0.01%)
agg["days_published_%"] = agg["days_published"] / n_days * 100
agg.sort_values("mean_share_bp", ascending=False).head(10)

## 3.2 Artefact check: "super-cells"
In advertising-SDK location data, devices whose real position is unknown are sometimes placed at a default
point (for example a city centroid). Such cells show an implausibly high and very stable share every day.
We flag cells whose mean share is far above the rest (robust z-score on the log share).

In [ ]:
log_share = np.log10(agg["mean_share_bp"].clip(lower=1e-3))
med = log_share.median()
mad = (log_share - med).abs().median()
agg["robust_z"] = (log_share - med) / (1.4826 * mad)
suspects = agg[(agg["robust_z"] > 6) & (agg["days_published_%"] > 95)].sort_values("mean_share_bp", ascending=False)
print(f"Flagged {len(suspects)} possible artefact cells")
suspects.head(10)

In [ ]:
# Stability over time of the top flagged cell vs a normal busy cell
if len(suspects):
    top_id = suspects.iloc[0]["cell_id"]
    normal_id = agg[~agg["cell_id"].isin(suspects["cell_id"])].nlargest(1, "mean_share_bp").iloc[0]["cell_id"]
    fig, ax = plt.subplots(figsize=(12, 4))
    for cid, lab in [(top_id, "flagged super-cell"), (normal_id, "busiest normal cell")]:
        ts = cells[cells["cell_id"] == cid].set_index("date")["share_day"] * 100
        ax.plot(ts.index, ts.values, label=f"{cid} ({lab})")
    ax.set_ylabel("% of daily presence")
    ax.set_title("A super-cell holds a large, steady share every day")
    ax.legend()
    jp.savefig(fig, "03_supercell_timeseries.png")
    plt.show()

clean = agg[~agg["cell_id"].isin(suspects["cell_id"])].copy()

## 3.3 Presence map (artefact cells removed)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sc = ax.scatter(clean["centroid_lon"], clean["centroid_lat"],
                c=np.log10(clean["mean_share_bp"].clip(lower=0.01)),
                cmap="magma_r", s=6, marker="s")
cb = plt.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label("log10 mean share of daily presence (bp)")
ax.set_aspect("equal")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Where people are: 500 m presence in DIY, Dec 2021 - Apr 2022")
jp.savefig(fig, "03_presence_map.png")
plt.show()

In [ ]:
# Interactive version on OpenStreetMap (open figures/03_presence_map.html in a browser)
fig_i = px.scatter_map(
    clean.assign(log_share=np.log10(clean["mean_share_bp"].clip(lower=0.01))),
    lat="centroid_lat", lon="centroid_lon", color="log_share",
    hover_data={"cell_id": True, "mean_share_bp": ":.2f", "days_published_%": ":.0f"},
    color_continuous_scale="Magma_r", zoom=9, height=700, map_style="open-street-map",
    title="Mean share of daily presence (bp) per 500 m cell")
fig_i.update_traces(marker={"size": 6})
fig_i.write_html(jp.FIG_DIR / "03_presence_map.html")

## 3.4 Holiday hotspots: which cells gain share during 24 Dec - 2 Jan?
`uplift = mean share in holiday window / mean share on ordinary weekdays (1-20 Dec)`
Only cells with a meaningful baseline (>= 3 bp) are ranked, to avoid noise from tiny counts.

In [ ]:
start, end = map(pd.Timestamp, jp.HOLIDAY_WINDOW)
hol_days = cells.loc[cells["date"].between(start, end), "date"].nunique()
base_mask = cells["date"].between("2021-12-01", "2021-12-20") & (cells["date"].dt.dayofweek < 5)
base_days = cells.loc[base_mask, "date"].nunique()

hol = cells[cells["date"].between(start, end)].groupby("cell_id")["share_day"].sum() / hol_days
base = cells[base_mask].groupby("cell_id")["share_day"].sum() / base_days
up = clean.set_index("cell_id").join(pd.DataFrame({"hol": hol, "base": base})).dropna(subset=["base"])
up = up[up["base"] * 1e4 >= 3].fillna({"hol": 0})
up["uplift"] = up["hol"] / up["base"]
top_up = up.sort_values("uplift", ascending=False).head(15)
top_up[["centroid_lon", "centroid_lat", "base", "hol", "uplift"]].assign(
    base=lambda d: (d["base"] * 1e4).round(1), hol=lambda d: (d["hol"] * 1e4).round(1)).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(clean["centroid_lon"], clean["centroid_lat"], s=4, c="lightgrey", marker="s")
sc = ax.scatter(up["centroid_lon"], up["centroid_lat"], c=np.log2(up["uplift"].clip(0.25, 4)),
                cmap="RdBu_r", vmin=-2, vmax=2, s=10, marker="s")
for _, r in top_up.head(8).reset_index().iterrows():
    ax.annotate(f"x{r['uplift']:.1f}", (r["centroid_lon"], r["centroid_lat"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
cb = plt.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label("log2 holiday uplift (red = busier in holidays)")
ax.set_aspect("equal")
ax.set_title("Holiday uplift: 24 Dec - 2 Jan vs ordinary December weekdays")
jp.savefig(fig, "03_holiday_uplift_map.png")
plt.show()

fig_u = px.scatter_map(up.reset_index(), lat="centroid_lat", lon="centroid_lon",
                       color=np.log2(up["uplift"].clip(0.25, 4)).values,
                       color_continuous_scale="RdBu_r", range_color=(-2, 2),
                       hover_data={"cell_id": True, "uplift": ":.2f"},
                       zoom=9, height=700, map_style="open-street-map",
                       title="Holiday uplift per 500 m cell (log2)")
fig_u.write_html(jp.FIG_DIR / "03_holiday_uplift_map.html")
top_up.reset_index().to_csv(jp.FIG_DIR / "03_top_holiday_cells.csv", index=False)

## Key findings (fill in after running)
- A handful of super-cells hold several percent of all daily presence with almost no day-to-day variation:
  likely default-location artefacts, so we exclude them from hotspot maps (good critical-thinking point for the presentation).
- Presence is concentrated along the Kota Yogyakarta - Sleman urban corridor.
- Holiday uplift is highest in ... (open `03_holiday_uplift_map.html`, hover over the top cells and name the places).

In [ ]:
# @title Unduh semua grafik (Colab)
# Membuat figures.zip lalu mengunduhnya ke laptop, untuk slide dan untuk di-push ke GitHub.
if IN_COLAB:
    import shutil
    from google.colab import files as colab_files
    shutil.make_archive("/content/figures", "zip", "/content/jogjapulse/figures")
    colab_files.download("/content/figures.zip")
